# Notebook 04 — Ablation Analysis
**Project**: IndicSenti — Multilingual Indian Sentiment Analysis  
Loads JSON results from 8 ablation studies and generates paper figures.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

PALETTE = {
    'mint': '#F1F6F4', 'gold': '#FFC801', 'teal': '#114C5A',
    'sage': '#D9E8E2', 'orange': '#FF9932', 'dark': '#172B36',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['mint'], 'axes.facecolor': PALETTE['mint'],
    'axes.titlecolor': PALETTE['teal'], 'axes.labelcolor': PALETTE['dark'],
    'text.color': PALETTE['dark'], 'grid.color': PALETTE['sage'],
    'figure.dpi': 120, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
})

RESULTS_DIR = Path('../training/ablation/results')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)
print('Results dir:', RESULTS_DIR)

In [ ]:
# ── Load all ablation JSON results ───────────────────────────
def load_study(name):
    path = RESULTS_DIR / f'{name}.json'
    if not path.exists():
        print(f'[warn] {name}.json not found — using mock data')
        return None
    with open(path) as f:
        data = json.load(f)
    return pd.DataFrame(data)

STUDY_NAMES = [
    'base_model', 'lora_rank', 'data_fraction', 'language_exclusion',
    'code_mix_handling', 'data_augmentation', 'loss_function', 'quantization_impact'
]

studies = {name: load_study(name) for name in STUDY_NAMES}
loaded = [k for k, v in studies.items() if v is not None]
print(f'Loaded {len(loaded)}/{len(STUDY_NAMES)} studies: {loaded}')

In [ ]:
# ── Mock data for offline notebook execution ─────────────────
# Replace with real results once training completes
MOCK_DATA = {
    'base_model': pd.DataFrame({
        'config': ['mBERT', 'XLM-RoBERTa', 'MuRIL', 'IndicBERT'],
        'macro_f1': [0.712, 0.783, 0.836, 0.851],
        'accuracy': [0.731, 0.797, 0.842, 0.858],
    }),
    'lora_rank': pd.DataFrame({
        'config': ['r=4', 'r=8', 'r=16', 'r=32', 'r=64'],
        'macro_f1': [0.813, 0.832, 0.851, 0.854, 0.855],
        'accuracy': [0.821, 0.840, 0.858, 0.861, 0.862],
        'trainable_params': [0.30, 0.60, 1.18, 2.36, 4.72],  # millions
    }),
    'data_fraction': pd.DataFrame({
        'config': ['10%', '25%', '50%', '75%', '100%'],
        'macro_f1': [0.721, 0.784, 0.827, 0.843, 0.851],
        'accuracy': [0.735, 0.796, 0.839, 0.851, 0.858],
    }),
    'language_exclusion': pd.DataFrame({
        'config': ['-Hindi', '-Tamil', '-Bengali', '-Marathi', '-Telugu', 'All'],
        'macro_f1': [0.762, 0.831, 0.839, 0.842, 0.840, 0.851],
        'accuracy': [0.774, 0.840, 0.848, 0.850, 0.848, 0.858],
    }),
    'code_mix_handling': pd.DataFrame({
        'config': ['none', 'lang_prefix', 'ascii_norm', 'script_sep'],
        'macro_f1': [0.837, 0.845, 0.841, 0.851],
        'accuracy': [0.843, 0.852, 0.848, 0.858],
    }),
    'data_augmentation': pd.DataFrame({
        'config': ['no_aug', 'BT_only', 'synonym_only', 'aug_combined'],
        'macro_f1': [0.838, 0.843, 0.840, 0.851],
        'accuracy': [0.845, 0.850, 0.847, 0.858],
    }),
    'loss_function': pd.DataFrame({
        'config': ['CrossEntropy', 'FocalLoss', 'LabelSmoothing', 'WeightedCE'],
        'macro_f1': [0.830, 0.851, 0.837, 0.842],
        'accuracy': [0.840, 0.858, 0.845, 0.849],
    }),
    'quantization_impact': pd.DataFrame({
        'config': ['FP32', 'FP16', 'INT8', 'NF4'],
        'macro_f1': [0.851, 0.851, 0.846, 0.849],
        'accuracy': [0.858, 0.858, 0.853, 0.856],
        'inference_ms': [142, 23, 31, 18],
        'memory_mb': [147, 74, 37, 18],
    }),
}
# Fill missing with mock
for name in STUDY_NAMES:
    if studies[name] is None:
        studies[name] = MOCK_DATA[name]
print('All studies ready.')

In [ ]:
# ── Fig 9: 2×4 ablation overview ─────────────────────────────
STUDY_TITLES = [
    'Base Model', 'LoRA Rank', 'Data Fraction', 'Lang. Exclusion',
    'Code-Mix Handling', 'Data Augmentation', 'Loss Function', 'Quantization',
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Figure 9 — Ablation Study Overview (Macro-F1)', fontsize=14,
             color=PALETTE['teal'], fontweight='bold')
axes = axes.flatten()

for ax, name, title in zip(axes, STUDY_NAMES, STUDY_TITLES):
    df = studies[name]
    f1s = df['macro_f1'].values
    best_idx = np.argmax(f1s)
    colors = [PALETTE['gold'] if i == best_idx else PALETTE['sage'] for i in range(len(f1s))]
    
    bars = ax.bar(df['config'], f1s, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', fontsize=7.5, fontweight='bold',
                color=PALETTE['dark'])
    ax.set_ylim(min(f1s)*0.98, max(f1s)*1.025)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', labelsize=7.5, rotation=20)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='y', alpha=0.4)
    ax.spines[['top','right']].set_visible(False)

# Legend
from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor=PALETTE['gold'], label='Best config'),
    Patch(facecolor=PALETTE['sage'], label='Other configs'),
], loc='lower center', ncol=2, framealpha=0.5, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('../reports/fig9_ablation_overview.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Fig 10: LoRA rank vs params trade-off ────────────────────
df_rank = studies['lora_rank']

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

x = range(len(df_rank))
ax1.plot(x, df_rank['macro_f1'], color=PALETTE['teal'], linewidth=2.5,
         marker='o', markersize=8, label='Macro-F1')
ax1.set_ylabel('Macro-F1', color=PALETTE['teal'])
ax1.tick_params(axis='y', labelcolor=PALETTE['teal'])

ax2.bar(x, df_rank['trainable_params'], alpha=0.3, color=PALETTE['orange'],
        label='Trainable Params (M)', width=0.4)
ax2.set_ylabel('Trainable Parameters (M)', color=PALETTE['orange'])
ax2.tick_params(axis='y', labelcolor=PALETTE['orange'])

ax1.set_xticks(x); ax1.set_xticklabels(df_rank['config'])
ax1.set_title('Figure 10 — LoRA Rank: F1 vs Parameter Count Trade-off')
ax1.grid(axis='y', alpha=0.3)
ax1.spines[['top']].set_visible(False)

# Sweet spot annotation
r16_idx = list(df_rank['config']).index('r=16')
ax1.annotate('★ Sweet spot\nr=16', xy=(r16_idx, df_rank['macro_f1'].iloc[r16_idx]),
              xytext=(r16_idx+0.5, df_rank['macro_f1'].iloc[r16_idx]-0.005),
              arrowprops=dict(arrowstyle='->', color=PALETTE['gold']),
              color=PALETTE['gold'], fontweight='bold')

fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.88))
plt.tight_layout()
plt.savefig('../reports/fig10_lora_rank_tradeoff.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 11: Quantization — F1 vs latency vs memory ──────────
df_q = studies['quantization_impact']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Figure 11 — Quantization Impact', color=PALETTE['teal'], fontweight='bold')

props = [('macro_f1', 'Macro-F1', PALETTE['teal']),
         ('inference_ms', 'Latency (ms)', PALETTE['gold']),
         ('memory_mb', 'Memory (MB)', PALETTE['orange'])]

for ax, (col, ylabel, color) in zip(axes, props):
    vals = df_q[col].values
    best_idx = (np.argmax if col == 'macro_f1' else np.argmin)(vals)
    clrs = [PALETTE['gold'] if i == best_idx else color for i in range(len(vals))]
    ax.bar(df_q['config'], vals, color=clrs, edgecolor='white')
    ax.set_title(ylabel)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(ax.patches, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                str(val), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/fig11_quantization_impact.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Full ablation summary table──────────────────────────────
rows = []
for name, title in zip(STUDY_NAMES, STUDY_TITLES):
    df = studies[name]
    best = df.loc[df['macro_f1'].idxmax()]
    rows.append({'Study': title, 'Best Config': best['config'],
                 'Best F1': best['macro_f1'],
                 'Δ vs Worst': best['macro_f1'] - df['macro_f1'].min()})

summary = pd.DataFrame(rows).set_index('Study')
summary['Best F1']   = summary['Best F1'].map('{:.3f}'.format)
summary['Δ vs Worst'] = summary['Δ vs Worst'].map('+{:.3f}'.format)
print(summary.to_string())